# Import Libraries & Datasets

The following code is an implementation of Chris Tran's guide on creating a CNN for sentence classification in Python. The following guide is linked here:

https://chriskhanhtran.github.io/posts/cnn-sentence-classification/

The implementation was made to be used on the following HuggingFace dataset:
https://huggingface.co/datasets/lmsys/chatbot_arena_conversations

In [1]:
import os
import re
from tqdm import tqdm
import numpy as np
import pandas as pd
import nltk
nltk.download("all")
import matplotlib.pyplot as plt
import torch

%matplotlib inline

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

In [2]:
# log into huggingface
!pip install huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

# Import Dataset
from datasets import load_dataset
ds = load_dataset("lmsys/chatbot_arena_conversations", split = "train")

# Turn dataset into pandas dataframe
ds = ds.to_pandas()

README.md:   0%|          | 0.00/7.00k [00:00<?, ?B/s]

data/train-00000-of-00001-cced8514c7ed78(…): reconstructing file:   0%|          |  0.00B / 41.6MB            

data/train-00000-of-00001-cced8514c7ed78(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/33000 [00:00<?, ? examples/s]

In [3]:
# Specify GPU for training (made in Google Colab)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f'There are {torch.cuda.device_count()} GPU(s) available.')
    print('Device name:', torch.cuda.get_device_name(0))

else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")


There are 1 GPU(s) available.
Device name: Tesla T4


# Data Preparation

In [1]:
# --------------------------------------------------------------------------
# Flattening dataset rows
# ---------------------------------------------------------------------------
def extract_conv_data(conversation):
    """
       conversation is a list of dictionaries like: [{"content": "...", "role": "user"}, {"content": "...", "role": "assistant"}, ...]
       Returns a tuple containing each conversation's user type and corresponding text.
       Grabs only the first exchange between user and assistant for simplicity.
    """
    user_text, assistant_text = None, None
    for turn in conversation:
        role = turn.get("role")
        content = turn.get("content", "")
        if role == "user" and user_text is None:
            user_text = content
        elif role == "assistant" and assistant_text is None:
            assistant_text = content
        if user_text is not None and assistant_text is not None:
            break
    return user_text, assistant_text


def flatten_df(df):
    """Split each row's conversation_a/conversation_b into two separate (model_name, input, output) records."""
    records = []
    for _, row in df.iterrows():
        for side in ("a", "b"):
            model_name = row[f"model_{side}"]
            conv = row[f"conversation_{side}"]
            user_text, assistant_text = extract_conv_data(conv)
            if user_text and assistant_text:
                records.append({
                    "model_name": model_name,
                    "llm_input": user_text,
                    "llm_output": assistant_text,
                })
    return pd.DataFrame(records)

# ---------------------------------------------------------------------------
# Filtering rows and balancing classes
# ---------------------------------------------------------------------------
def filter_and_balance(df, models, min_count=None, random_state=42):
    """
       Filters out rows according to given model names. Also removes duplicate and NULL rows.
       Balances the number of examples for each type of model class to be used in the cleaned dataset.
    """
    df = df[df["model_name"].isin(models)].copy()
    df = df.drop_duplicates(subset=["llm_input", "llm_output", "model_name"])
    df = df[df["llm_output"].str.strip().str.len() > 0]

    counts = df["model_name"].value_counts()
    print("Counts before balancing:\n", counts)

    target_n = min_count if min_count else counts.min()
    balanced = (
        df.groupby("model_name", group_keys=False)
        .apply(lambda g: g.sample(min(len(g), target_n), random_state=random_state))
        .reset_index(drop=True)
    )
    print("\nCounts after balancing:\n", balanced["model_name"].value_counts())
    return balanced


# ---------------------------------------------------------------------------
# Cleaning & Tokenizing text
# ---------------------------------------------------------------------------
def clean_str(text):
    """
       Chris Tran's guide cleaned the text of his input, so I followed suit here.
       This function was copied from Tran's guide, except for the first 3 lines.
       These 3 lines are additional cleaning lines added for my specific dataset's purpose.
    """
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)                  # strip HTML tags
    text = re.sub(r"```.*?```", " ", text, flags=re.S)    # strip code fences
    text = re.sub(r"http\S+|www\.\S+", " ", text)         # strip URLs
    text = re.sub(r"[^A-Za-z0-9(),!?\'\`]", " ", text)
    text = re.sub(r"\'s", " \'s", text)
    text = re.sub(r"\'ve", " \'ve", text)
    text = re.sub(r"n\'t", " n\'t", text)
    text = re.sub(r"\'re", " \'re", text)
    text = re.sub(r"\'d", " \'d", text)
    text = re.sub(r"\'ll", " \'ll", text)
    text = re.sub(r",", " , ", text)
    text = re.sub(r"!", " ! ", text)
    text = re.sub(r"\(", " ( ", text)
    text = re.sub(r"\)", " ) ", text)
    text = re.sub(r"\?", " ? ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip().lower()


def tokenize(text):
    """Turns cleaned text into a list of tokens, tokenized based on whitespace"""
    return clean_str(text).split()


In [ ]:
# Tokenize, build vocabulary, encode tokens
print("Tokenizing...\n")
tokenized_texts, word2idx, max_len = tokenize(texts)
input_ids = encode(tokenized_texts, word2idx, max_len)

# Load pretrained vectors
embeddings = load_pretrained_vectors(word2idx, "fastText/crawl-300d-2M.vec")
embeddings = torch.tensor(embeddings)